# Create a before/after-map of OpenStreetMap data

This notebook allows you to make a set of images of OSM data of an area from 2 dates, showing what was changed, making use of the [osm-mapping-party-before-after](https://github.com/amandasaurus/osm-mapping-party-before-after/) tool.

## Getting started

### If this is your first time using a notebook like this: 

1. You can use the up/down arrow keys of your keyboard to move between the different "cells" or regions of this notebook. The currently selected one will be highlighed on the left with a colour bar.
2. You can "run" each of these cells/regions by either pressing the "play" button in the menu above, or by pressing CTRL+ENTER on your keyboard.
3. If you are in the "editing" mode where you type into a cell, pressing ESC will take you back to the mode where the arrow keys work. 

## Getting an OSM history file

Before we can do the rest of the steps in this notebook, you will have to upload an OSM history file that covers your region of interest. You can find the corresponding files on [Geofabrik's internal download server](https://osm-internal.download.geofabrik.de/?landing_page=true). Login with your OSM account to get to the `internal` files, and then find the page for the your region of interest. Ideally, find the smallest region that covers your area of interest to minimize the filesize you need to upload. 

**The right file**: When you've picked your region, you will likely see two similarly named files listed: `regionname-internal.osh.pbf` and `regionname-latest-internal.osm.pbf`. The file you will want to download is the `regionname-internal.osh.pbf` (i.e. the one that includes `osh` and **no `latest`**).

Once you have downloaded your file onto your computer, you need to upload it into your container. There are a few options that you can consider, depending on how you are using this notebook:

### 1. Uploading a file you already have downloaded onto your computer

* If you've launched this notebook in _MyBinder_, you can then open up the filebrowser on the top of the left hand side (The _file_ icon, or by pressing CTRL+SHIFT+F). 
* If you've launched this notebook locally, you're likely in the _notebook_ interface, switch back to the _tree_ view and upload your file there (or put it into the shared folder which is accessible by both the container and your normal operating system).

### 2. Downloading a file directly from Geofabrik 

This option is particularly useful if you use this notebook on MyBinder, as it allows you to directly download the OSM history file into your Binder, without having to first download it onto your computer and then re-uploading it to the Binder, speeding things up considerably. As the OSM history files are only accessible with an OSM account, you will have to be authenticated to use this method. 

There are two ways of achieving this to access files directly from _Geofabrik_. 

#### 2.1 Option a): Using your OSM username & password.

You can use your OSM username & password to authenticate, by running the cell below and then entering the form fields. 

**Note:** Your username & password will only be used to get a cookie that allows downloading the files and won't be permanently stored (and are not accessible to anyone else). If you are using _MyBinder_, your virtual machine will automatically shut down after more than 10 minutes of inactivity, destroying all data within the virtual machine in the process. If you are running this in a Docker container on your own computer the container will also destroy itself after shutting it down. 

If you decide to use the direct download from Geofabrik, enter the necessary details here: 

In [1]:
import ipywidgets as widgets

osm_user = widgets.Text(
    placeholder='Steve',
    description='OSM username:',
    disabled=False   
)

osm_password = widgets.Text(
    placeholder='your password',
    description='OSM password:',
    disabled=False   
)

file_url = widgets.Text(
    placeholder='https://osm-internal.download.geofabrik.de/south-america/argentina-internal.osh.pbf',
    description='Link to OSM history file:',
    disabled=False   
)
widgets.VBox([
    osm_user,
    osm_password,
    file_url
    ])

Once you have added the details above **(Note: you don't have to re-run the cell that creates this form after filling in the information!)**, you can download your file by running the cell below this one:

In [ ]:
!python sendfile_osm_oauth_protector/oauth_cookie_client.py -u "{osm_user.value}" -p "{osm_password.value}" -c "https://osm-internal.download.geofabrik.de/get_cookie" -f netscape -o geofabrik_cookie.txt;wget --max-redirect 0 --load-cookies geofabrik_cookie.txt {file_url.value}

Once this is finished, you can move down to **Starting to make comparison images**!

#### 2.1 Option b): Using an existing Geofabrik cookie from your browser. 

This method allows you to paste the cookie that Geofabrik uses to check if you're allowed to download a file. As a result, **this method does not need to enter your OSM username/password**. The downside is that it's slightly more complicated, as it requires finding the cookie value in your web browser. 

To use this option, do the following: 

1. Head to the [OSM internal Geofabrik download page](https://osm-internal.download.geofabrik.de/)
2. Log in on that page with your existing OSM credentials, this will create the necessary cookie. 
3. Find the cookie in your web browser: If you're using Firefox, press `F12` on your keyboard to open the web developer tools
4. Go to the tab called `Storage` in the bar that opens at the bottom of your browser
5. On the bottom left, you will see `Cookies`, click on it and select the cookie that starts with `https://osm-internal.download.geofabrik.de`.
6. You will see a Name/Value pair displayed, with the name reading `gf_download_oauth`. Select this cookie
7. On the right-hand side you will see a `Data` option, which gives the full cookie contents. You can right-click on this and select `Copy`.

You can then paste this cookie data into the form below, alongside the link to the file you want to download:

In [2]:
geofabrik_cookie = ('gf_download_oauth', widgets.Text(
    placeholder='login|2018-04-12|WVRced34c...',
    description='gf_download_oauth cookie value:',
    disabled=False))
file_url = widgets.Text(
    placeholder='https://osm-internal.download.geofabrik.de/south-america/argentina-internal.osh.pbf',
    description='Link to OSM history file:',
    disabled=False)

widgets.VBox([
    geofabrik_cookie[1],
    file_url])

Once you have added those, run the cell below to download your file. **(Reminder: you don't have to re-run the cell that creates the form after filling in the information!)**

In [ ]:
cookie_name = geofabrik_cookie[0]
if geofabrik_cookie[1].value.startswith("gf_download_oauth"):
    cookie_value = geofabrik_cookie[1].value.replace('gf_download_oauth:"','')
    cookie_value = cookie_value[:-1]
    print(cookie_value)
else:
    cookie_value = geofabrik_cookie[1].value
    
!wget --max-redirect 0 --header="Cookie: {cookie_name}={cookie_value}" {file_url.value}

Once the file is completely downloaded (or if you uploaded the file using one of the other methods above), you can continue with the next step below. 

## Starting to make the comparison images. 

When you run the code-cell below, it will generate some "forms" that will feel familiar form other web-forms, which you can use to enter all the necessary parameters to create your images. These forms allow you to add the _history file_ you have just uploaded, the two time-points you want to compare, the bounding box (i.e. the area you're interested in, which you can find e.g. [via bboxfinder](https://bboxfinder.com)) and the minimum/maximum zoom levels, if you want to create images at different zoom levels. 

Now, run the cell below and then enter all the necessary information in the forms:

In [ ]:
import ipywidgets as widgets
import glob

filename = widgets.Dropdown(
    options=glob.glob("**/*.pbf",recursive=True),
    description='File:',
    disabled=False,
)

bbox_label= widgets.Label(
    value="BBox is in long/lat and can be found via bboxfinder.com",
    layout=widgets.Layout(width='100%')
)

bbox = widgets.Text(
    placeholder='-61.975228,-33.749067,-61.953167,-33.733014',
    description='BBox:',
    disabled=False   
)


start_date = widgets.NaiveDatetimePicker(description='Start datetime')
end_date = widgets.NaiveDatetimePicker(description='End datetime')

date_label= widgets.Label(
    value="You need to pick a time for the script to work, even if it's just 00:00",
    layout=widgets.Layout(width='100%')
)

min_zoom_level = widgets.Dropdown(
    options=range(0,21),
    description='Min Zoom:',
    disabled=False,
)

max_zoom_level = widgets.Dropdown(
    options=range(0,21),
    description='Max Zoom:',
    disabled=False,
)

num_frames = widgets.BoundedIntText(
    min=2,
    step=1,
    value=2,
    description='Number of frames:',
    disabled=False
)

widgets.VBox([
    filename,
    bbox,
    bbox_label,
    start_date,
    end_date,
    date_label,
    min_zoom_level,
    max_zoom_level,
    num_frames])

Once you have set your parameters, you can run the cell below in the same way that you ran the cell above for creating the forms **(Reminder: you don't have to re-run the cell that creates the form after filling in the information!)**. 

Running the cell below will take a while, in particular if it's the first time you execute it, as it needs to download some external resources first. Subsequent runs will be faster. You can re-run it as many times as you want, and change any of the parameters using the forms above. 

Once you run the cell below, it will print a lot of output into this notebook, including some warnings. If everything works, the last things printed out will read *Generating comparison image for zoom XXX*, according to the zoom levels you picked. 

The resulting images will be saved as "progress.filename.*parameters*.png/gif.

In [ ]:
!./make.sh {filename.value} {start_date.value.isoformat()}Z {end_date.value.isoformat()}Z {bbox.value} {min_zoom_level.value} {max_zoom_level.value} {num_frames.value}

Enjoy mapping and visualizing!

If this all worked you should be able to see the output files in the filebrowser/file tree, named `progress.filename.parameters.png/gif.`. 

If you are running this in the MyBinder, you should download them to your computer to keep them permanently.

If you are running this locally, you can do this as well or alternatively move them into the shared "output" folder that is accessible from your host computer.  

